# Correlation Analysis

In this notebook, we analyze the relationships between different commodities to see if, for instance, energy prices (Oil) drive agricultural prices (Food).

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 10)

In [ ]:
conn = sqlite3.connect('../sql/commodity_data.db')
query = """
SELECT 
    p.date, 
    c.commodity_name, 
    p.price_nominal_usd
FROM prices p 
JOIN commodities c ON p.commodity_id = c.commodity_id
"""

df = pd.read_sql_query(query, conn)
df['date'] = pd.to_datetime(df['date'])
conn.close()

# Pivot to get dates on index and commodities on columns
pivot_df = df.groupby(['date', 'commodity_name'])['price_nominal_usd'].mean().unstack()
pivot_df.head()

## 1. Correlation Heatmap
We select a diverse set of representative commodities across categories: Energy, Precious Metals, Agriculture.

In [ ]:
commodities_to_correlate = [
    'Crude oil, average', 'Natural gas, US', 'Coal, Australian',
    'Gold', 'Silver', 'Copper', 'Aluminum',
    'Wheat, US SRW', 'Maize', 'Soybeans', 'Sugar, world'
]

subset_pivot = pivot_df[commodities_to_correlate].dropna()
corr_matrix = subset_pivot.corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', vmin=-1, vmax=1, fmt=".2f", linewidths=.5)
plt.title('Commodity Price Correlation Matrix (Pearson)', fontsize=16)
plt.savefig('../images/correlation_heatmap.png', bbox_inches='tight')
plt.show()